In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [4]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

#region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [6]:
years = ['2019', '2020', '2021', '2022', '2023', '2024']
year_dropdown = widgets.Dropdown(
    options=years,
    value='2020',
    description='Select Year:',
    style={'description_width': 'initial'}
)

month_names = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']

month_dropdown = widgets.Dropdown(
    options=month_names,
    value='January',
    description='Select Month:',
    style={'description_width': 'initial'}
)

output_widget = widgets.Output()

In [7]:
def update_map(change, map, region, year_dropdown, month_dropdown, output_widget):
    selected_year = int(year_dropdown.value)
    selected_month_name = month_dropdown.value
    selected_month = month_names.index(selected_month_name) + 1
    
    with output_widget:
        output_widget.clear_output()
        print(f"Loading data for: {selected_year} - {selected_month_name}")

    # cleaning previous layers
    if map.find_layer('Monsoon Radar'):
        map.remove_layer('Monsoon Radar')
    if map.find_layer('Region Boundary'):
        map.remove_layer('Region Boundary')

    # defining dates
    last_day = 30
    if selected_month == 2:
        last_day = 28
    
    start_date = ee.Date.fromYMD(selected_year, selected_month, 1)
    end_date = ee.Date.fromYMD(selected_year, selected_month, last_day)

    # loading rader data
    sar_image = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(region) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .median() \
        .clip(region)

    vis_params = {'min': -25, 'max': -5}
    layer_name = 'Monsoon Radar'
    map.addLayer(sar_image, vis_params, layer_name)

    empty = ee.Image().byte()
    outline = empty.paint(featureCollection=ee.FeatureCollection(region), color=1, width=2)
    map.addLayer(outline, {'palette': 'red'}, 'Region Boundary')

    return 



In [10]:
year_dropdown.observe(lambda change: update_map(change, map, region, year_dropdown, month_dropdown, output_widget), names='value')
month_dropdown.observe(lambda change: update_map(change, map, region, year_dropdown, month_dropdown, output_widget), names='value')
ui = widgets.VBox([year_dropdown, month_dropdown, output_widget])
display(ui)
map

Map(bottom=226828.0, center=[23.745125865762933, 90.40580749511719], controls=(WidgetControl(options=['positio…